In [1]:
import pandas as pd
import numpy as np
from scipy.optimize import minimize
from codecarbon import EmissionsTracker
from datetime import timedelta
import warnings
warnings.filterwarnings("ignore")

In [2]:
# ==========================================
# 1. PREPARATION DES DONNEES (Data Prep)
# ==========================================
def prepare_hawkes_data(filepath):
    """
    Charge et transforme le CSV en séquences temporelles par alerte.
    """
    print("Chargement des données...")
    df = pd.read_csv(filepath)
    df['date'] = pd.to_datetime(df['date'])
    
    # On filtre pour ne garder que les éclairs faisant partie d'une alerte
    df = df.dropna(subset=['airport_alert_id']).copy()
    df = df.sort_values(['airport_alert_id', 'date'])
    
    # Extraction des séquences temporelles
    alerts_data = {}
    grouped = df.groupby('airport_alert_id')
    
    for alert_id, group in grouped:
        # On convertit les timestamps en minutes écoulées depuis le 1er éclair
        t0 = group['date'].min()
        t_events = (group['date'] - t0).dt.total_seconds().values / 60.0
        
        # T_max est l'heure de fin d'observation (on ajoute 30 min après le dernier)
        T_max = t_events[-1] + 30.0 
        
        alerts_data[alert_id] = {
            't_events': t_events,
            'T_max': T_max,
            'year': t0.year
        }
    return alerts_data

In [3]:
# ==========================================
# 2. LE MOTEUR MATHEMATIQUE (Hawkes)
# ==========================================
def hawkes_joint_log_likelihood(params, alerts_list):
    """
    Calcule la Log-Vraisemblance globale sur un ensemble d'alertes.
    """
    mu, alpha, beta = params
    
    # Contraintes de stabilité physique
    if mu <= 1e-5 or alpha <= 1e-5 or beta <= 1e-5 or alpha >= beta:
        return 1e9 
    
    total_ll = 0.0
    for alert in alerts_list:
        t_events = alert['t_events']
        T_max = alert['T_max']
        n = len(t_events)
        
        if n == 0:
            total_ll -= (mu * T_max)
            continue
            
        # Calcul récursif de l'intensité (O(N) par alerte)
        A = np.zeros(n)
        for i in range(1, n):
            A[i] = np.exp(-beta * (t_events[i] - t_events[i-1])) * (1 + A[i-1])
            
        intensities = mu + alpha * A
        log_lambda_sum = np.sum(np.log(intensities))
        
        integral_lambda = mu * T_max + (alpha / beta) * np.sum(1 - np.exp(-beta * (T_max - t_events)))
        
        total_ll += (log_lambda_sum - integral_lambda)
        
    return -total_ll # scipy minimise, donc on inverse


In [4]:
class HawkesModel:
    def __init__(self):
        self.params = [0.01, 0.5, 1.0] # [mu, alpha, beta]
        
    def fit(self, alerts_list):
        bounds = [(1e-4, 1.0), (1e-4, 5.0), (1e-4, 10.0)]
        print(f"Entraînement MLE sur {len(alerts_list)} alertes...")
        res = minimize(
            hawkes_joint_log_likelihood, 
            self.params, 
            args=(alerts_list,),
            method='L-BFGS-B', 
            bounds=bounds
        )
        self.params = res.x
        print(f"Paramètres appris : mu={self.params[0]:.4f}, alpha={self.params[1]:.4f}, beta={self.params[2]:.4f}")
        return self

    def predict_safe_proba(self, t_current, t_events, tau_horizon=30):
        mu, alpha, beta = self.params
        past_events = t_events[t_events <= t_current]
        
        Lambda_residual = mu * tau_horizon
        if len(past_events) > 0:
            for t_i in past_events:
                Lambda_residual += (alpha / beta) * (
                    np.exp(-beta * (t_current - t_i)) - 
                    np.exp(-beta * (t_current + tau_horizon - t_i))
                )
        return np.exp(-Lambda_residual)



In [8]:
# ==========================================
# 3. PIPELINE : ENTRAINEMENT ET EVALUATION
# ==========================================
def evaluate_model(model, test_alerts):
    print("\nÉvaluation du gain de temps (Lead Time)...")
    gains_minutes = []
    violations = 0
    
    for alert_id, alert in test_alerts.items():
        t_events = alert['t_events']
        if len(t_events) < 2: continue
            
        t_last_real = t_events[-1]
        regle_30_min = t_last_real + 30.0 # Règle actuelle [cite: 94]
        
        # On simule le temps qui passe minute par minute après le pic de l'orage
        t_reprise_ia = regle_30_min
        for t_simul in np.arange(t_events[0], regle_30_min, 1.0):
            p_safe = model.predict_safe_proba(t_simul, t_events, tau_horizon=30)
            
            # Si le modèle est sûr à 95% qu'il n'y aura plus d'éclair
            if p_safe > 0.95:
                # Vérification de sécurité : un éclair frappe-t-il APRES cette décision ?
                eclairs_futurs = t_events[t_events > t_simul]
                if len(eclairs_futurs) > 0:
                    violations += 1
                else:
                    t_reprise_ia = t_simul
                break
                
        gain = regle_30_min - t_reprise_ia
        if gain > 0:
            gains_minutes.append(gain)
            
    avg_gain = np.mean(gains_minutes) if gains_minutes else 0
    print("Gains :", gains_minutes)
    print(f"--> Gain de temps moyen validé : {avg_gain:.1f} minutes par alerte.")
    print(f"--> Violations de sécurité (Faux Positifs) : {violations}")
    return avg_gain, violations


In [9]:
# --- EXECUTION PRINCIPALE ---
if __name__ == "__main__":
    # 1. Chargement
    # Remplacez par le nom de votre fichier
    all_alerts = prepare_hawkes_data('segment_alerts_all_airports_train.csv')
    
    # 2. Partition Chronologique
    train_alerts = [a for a in all_alerts.values() if a['year'] <= 2023]
    test_alerts = {k: a for k, a in all_alerts.items() if a['year'] >= 2024}
    
    # 3. Entraînement avec suivi Carbone
    tracker = EmissionsTracker()
    tracker.start()
    
    model = HawkesModel()
    # Pour le prototype, on s'entraîne sur un échantillon (ex: 500 alertes) pour la rapidité
    model.fit(train_alerts) 
    
    tracker.stop()
    
    # 4. Évaluation Métier
    evaluate_model(model, test_alerts)

Chargement des données...


[codecarbon WARNING @ 00:43:21] Multiple instances of codecarbon are allowed to run at the same time.
[codecarbon WARNING @ 00:43:21] Error while trying to count physical CPUs: [Errno 2] No such file or directory: 'lscpu'. Defaulting to 1.
[codecarbon INFO @ 00:43:21] [setup] RAM Tracking...
[codecarbon INFO @ 00:43:21] [setup] CPU Tracking...
[codecarbon WARNING @ 00:43:21] No CPU tracking mode found. Falling back on estimation based on TDP for CPU. 
 Mac OS and ARM processor detected: Please enable PowerMetrics sudo to measure CPU

[codecarbon INFO @ 00:43:21] CPU Model on constant consumption mode: Apple M1
[codecarbon WARNING @ 00:43:21] No CPU tracking mode found. Falling back on CPU constant mode.
[codecarbon INFO @ 00:43:21] [setup] GPU Tracking...
[codecarbon INFO @ 00:43:21] No GPU found.
[codecarbon INFO @ 00:43:21] The below tracking methods have been set up:
                RAM Tracking Method: RAM power estimation model
                CPU Tracking Method: global constant


Entraînement MLE sur 769 alertes...


[codecarbon INFO @ 00:43:28] Energy consumed for RAM : 0.000003 kWh. RAM Power : 3.0 W
[codecarbon INFO @ 00:43:28] Delta energy consumed for CPU with constant : 0.000006 kWh, power : 5.0 W
[codecarbon INFO @ 00:43:28] Energy consumed for All CPU : 0.000006 kWh
[codecarbon INFO @ 00:43:28] 0.000009 kWh of electricity and 0.000000 L of water were used since the beginning.


Paramètres appris : mu=0.0100, alpha=0.5000, beta=1.0000

Évaluation du gain de temps (Lead Time)...
Gains : []
--> Gain de temps moyen validé : 0.0 minutes par alerte.
--> Violations de sécurité (Faux Positifs) : 0
